# DCCD model comparison table

`build_model_comparison_table(run_id)` builds one row per model under `results/<run_id>/` with:

- Structure adherence (`valid_structure_count/bt_count`) summed over all tasks
- Structure adherence % (2 dp)
- Task completion (`tasks_completed/total_tasks`)
- Task completion % (2 dp)
- Avg inference time (seconds)

For DCCD runs, `avg_inference_time_s` reflects planner + translator average inference time.

In [3]:
import sys
import json
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.utils import get_results_dir


def build_model_comparison_table(run_id: str) -> pd.DataFrame:
    results_dir = get_results_dir(run_id)
    rows = []
    for model_dir in sorted(p for p in results_dir.iterdir() if p.is_dir()):
        main_path = model_dir / "main_results.json"
        if not main_path.exists():
            continue

        main = json.loads(main_path.read_text(encoding="utf-8"))
        all_tasks = main.get("all_tasks", [])
        if not all_tasks:
            continue

        valid_structure_count = sum(int(task.get("valid_structure_count", 0)) for task in all_tasks)
        bt_count = sum(int(task.get("bt_count", 0)) for task in all_tasks)
        adherence_ratio = f"{valid_structure_count}/{bt_count}" if bt_count else "0/0"

        tasks_completed = sum(1 for task in all_tasks if bool(task.get("task_completion", False)))
        total_tasks = len(all_tasks)

        tasks_completed = int(main.get("total_task_completion", tasks_completed))
        total_tasks = int(main.get("total_tasks", total_tasks))
        task_completion_ratio = f"{tasks_completed}/{total_tasks}" if total_tasks else "0/0"

        rows.append({
            "model_id": main.get("model_id", model_dir.name),
            "Structure adherence": adherence_ratio,
            "Structure adherence %": f"{float(main.get('structure_adherence_pct', 0.0)):.2f}%",
            "Task completion": task_completion_ratio,
            "Task completion %": f"{float(main.get('task_completion_pct', 0.0)):.2f}%",
            "avg_inference_time": round(float(main.get("avg_inference_time_s", 0.0)), 3),
        })

    columns = [
        "model_id",
        "Structure adherence",
        "Structure adherence %",
        "Task completion",
        "Task completion %",
        "avg_inference_time",
    ]
    if not rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(rows, columns=columns).sort_values("model_id").reset_index(drop=True)


def build_partial_model_comparison_table(run_id: str) -> pd.DataFrame:
    results_dir = get_results_dir(run_id)
    rows = []

    for model_dir in sorted(p for p in results_dir.iterdir() if p.is_dir()):
        main_path = model_dir / "main_results.json"
        tasks_dir = model_dir / "tasks"

        run_status = "complete"
        model_id = model_dir.name
        all_tasks = []
        main = {}

        if main_path.exists():
            main = json.loads(main_path.read_text(encoding="utf-8"))
            all_tasks = main.get("all_tasks", [])
            model_id = main.get("model_id", model_dir.name)

        elif tasks_dir.exists():
            run_status = "partial"

            for task_path in sorted(tasks_dir.glob("*.json")):
                try:
                    task_result = json.loads(task_path.read_text(encoding="utf-8"))
                    task_result["task_id"] = task_path.stem
                    all_tasks.append(task_result)
                except Exception as exc:
                    print(f"Skipping unreadable task file: {task_path} ({exc})")

        else:
            continue

        if not all_tasks:
            continue

        valid_structure_count = sum(int(task.get("valid_structure_count", 0)) for task in all_tasks)
        bt_count = sum(int(task.get("bt_count", 0)) for task in all_tasks)
        adherence_pct = (100.0 * valid_structure_count / bt_count) if bt_count else 0.0

        tasks_completed = sum(1 for task in all_tasks if bool(task.get("task_completion", False)))
        total_tasks = len(all_tasks)
        task_completion_pct = (100.0 * tasks_completed / total_tasks) if total_tasks else 0.0

        avg_times = [
            float(task.get("avg_inference_time_s", 0.0))
            for task in all_tasks
            if task.get("avg_inference_time_s") is not None
        ]
        avg_inference_time = (sum(avg_times) / len(avg_times)) if avg_times else 0.0

        if main:
            tasks_completed = int(main.get("total_task_completion", tasks_completed))
            total_tasks = int(main.get("total_tasks", total_tasks))
            task_completion_pct = float(main.get("task_completion_pct", task_completion_pct))
            adherence_pct = float(main.get("structure_adherence_pct", adherence_pct))
            avg_inference_time = float(main.get("avg_inference_time_s", avg_inference_time))

        rows.append({
            "model_id": model_id,
            "run_status": run_status,
            "tasks_found": total_tasks,
            "Structure adherence": f"{valid_structure_count}/{bt_count}" if bt_count else "0/0",
            "Structure adherence %": f"{adherence_pct:.2f}%",
            "Task completion": f"{tasks_completed}/{total_tasks}" if total_tasks else "0/0",
            "Task completion %": f"{task_completion_pct:.2f}%",
            "avg_inference_time": round(avg_inference_time, 3),
        })

    columns = [
        "model_id",
        "run_status",
        "tasks_found",
        "Structure adherence",
        "Structure adherence %",
        "Task completion",
        "Task completion %",
        "avg_inference_time",
    ]
    if not rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(rows, columns=columns).sort_values("model_id").reset_index(drop=True)


In [4]:
from IPython.display import display, Markdown

run_id = "dccd_bt_all_tasks"

display(Markdown("## Complete Runs Only"))
display(build_model_comparison_table(run_id))

display(Markdown("## Complete + Partial Runs"))
display(build_partial_model_comparison_table(run_id))


## Complete Runs Only

,model_id,Structure adherence,Structure adherence %,Task completion,Task completion %,avg_inference_time
0,Qwen/Qwen2.5-14B-Instruct,279/288,96.88%,8/100,8.00%,13.517
1,Qwen/Qwen2.5-3B-Instruct,177/286,61.89%,13/100,13.00%,72.642
2,Qwen/Qwen2.5-7B-Instruct,267/268,99.63%,19/100,19.00%,6.509
3,Qwen/Qwen3-14B,291/291,100.00%,8/100,8.00%,10.725
4,Qwen/Qwen3-8B,281/283,99.29%,11/100,11.00%,8.878
5,ibm-granite/granite-3.3-8b-instruct,274/280,97.86%,14/100,14.00%,15.095
6,meta-llama/Llama-3.1-8B-Instruct,216/288,75.00%,6/100,6.00%,29.351


## Complete + Partial Runs

,model_id,run_status,tasks_found,Structure adherence,Structure adherence %,Task completion,Task completion %,avg_inference_time
0,Llama-3.2-3B-Instruct,partial,52,88/140,62.86%,8/52,15.38%,39.376
1,Qwen/Qwen2.5-14B-Instruct,complete,100,279/288,96.88%,8/100,8.00%,13.517
2,Qwen/Qwen2.5-3B-Instruct,complete,100,177/286,61.89%,13/100,13.00%,72.642
3,Qwen/Qwen2.5-7B-Instruct,complete,100,267/268,99.63%,19/100,19.00%,6.509
4,Qwen/Qwen3-14B,complete,100,291/291,100.00%,8/100,8.00%,10.725
5,Qwen/Qwen3-8B,complete,100,281/283,99.29%,11/100,11.00%,8.878
6,ibm-granite/granite-3.3-8b-instruct,complete,100,274/280,97.86%,14/100,14.00%,15.095
7,meta-llama/Llama-3.1-8B-Instruct,complete,100,216/288,75.00%,6/100,6.00%,29.351
